In [1]:
from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image
%matplotlib inline
ROOT = Path.cwd().parent
DATA = ROOT / 'data'
OUT = ROOT / 'results'
OUT.mkdir(exist_ok=True)

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

csv_path = DATA/'M29.csv'
df = pd.read_csv(csv_path)
# Identify filters by filename, independently of row order.
b, r, v = [df[df['Label'].str.contains('M29_' + band + '_')].iloc[0] for band in ['B', 'R', 'V']]
star_ids = [1, 2, 3, 4, 5, 6, 7, 8, 9, 12]

rows = []
for star in star_ids:
    suffix = 'T1' if star == 1 else f'C{star}'
    C_B, C_R, C_V = (float(row[f'Source-Sky_{suffix}']) for row in (b, r, v))
    E_B, E_R, E_V = (float(row[f'Source_Error_{suffix}']) for row in (b, r, v))
    bv = -2.5 * np.log10(C_B / C_V)
    vr = -2.5 * np.log10(C_V / C_R)
    k = 2.5 / np.log(10)
    bv_err = k * np.hypot(E_B / C_B, E_V / C_V)
    vr_err = k * np.hypot(E_V / C_V, E_R / C_R)
    rows.append((star, bv, bv_err, vr, vr_err))

results = pd.DataFrame(rows, columns=['Star', 'B-V', 'B-V error', 'V-R', 'V-R error'])
results.to_csv(OUT/'m29_instrumental_colors.csv', index=False)

fig, ax = plt.subplots(figsize=(12, 6))
ax.errorbar(results['B-V'], results['V-R'],
            xerr=results['B-V error'], yerr=results['V-R error'],
            fmt='o', capsize=3, markersize=6, linewidth=1, label='M29 stars')
for _, row in results.iterrows():
    ax.annotate(str(int(row['Star'])), (row['B-V'], row['V-R']),
                xytext=(6, 5), textcoords='offset points', fontsize=9)
ax.set(xlabel='Instrumental B−V (mag)', ylabel='Instrumental V−R (mag)',
       title='M29 Instrumental Color–Color Diagram')
ax.set_xlim(0.75, 1.60)
ax.set_ylim(0.15, 0.62)
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(OUT/'m29_color_color_original_style.pdf', bbox_inches='tight')
fig.savefig(OUT/'m29_color_color_original_style.png', dpi=220, bbox_inches='tight')
print(results.round(4).to_string(index=False))

 Star    B-V  B-V error    V-R  V-R error
    1 0.8167     0.0032 0.1965     0.0024
    2 0.9280     0.0033 0.3128     0.0023
    3 0.8994     0.0049 0.2916     0.0034
    4 0.8836     0.0036 0.2917     0.0025
    5 0.9010     0.0047 0.2628     0.0033
    6 1.0650     0.0038 0.3140     0.0024
    7 0.8092     0.0060 0.1892     0.0045
    8 1.5290     0.0077 0.5715     0.0034
    9 1.0900     0.0042 0.3808     0.0026
   12 0.9298     0.0033 0.2463     0.0023
